In [ ]:
from typing import List, Dict
from pydantic import BaseModel,Field
from langchain.agents import create_agent
from langchain_core.output_parsers import PydanticOutputParser
from langchain_ollama import ChatOllama
# 1. Define the output schema for parsing


questions = [
{"name": "impact_climate_biodiversity", "question": "This papers talks about the impact of climate change on biodiversity"},
{"name": "impact_climate_human_scoiety", "question": "This papers talks about the impact of climate change on human society"}
]
    
questions_str = "\n".join(
    [f"{q['name']}: {q['question']}" for q in questions]
)


In [73]:
class QuestionAnswer(BaseModel):
    answer: bool = Field(description="Binary anwser of the assessment") # True of False
    explanation: str = Field(description="Explanation of the binary answer in one or two sentences")  # 1-2 sentences


parser = PydanticOutputParser(pydantic_object=QuestionAnswer)

system_prompt = f"""
You are a precise assistant.
For the following assessment, answer with a JSON object containing:
- 'answer': 'True' or 'False'
- 'explanation': a short explanation (1-2 sentences)

Format: {parser.get_format_instructions()}
"""
print(system_prompt)


You are a precise assistant.
For the following assessment, answer with a JSON object containing:
- 'answer': 'True' or 'False'
- 'explanation': a short explanation (1-2 sentences)

Format: The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"answer": {"description": "Binary anwser of the assessment", "title": "Answer", "type": "boolean"}, "explanation": {"description": "Explanation of the binary answer in one or two sentences", "title": "Explanation", "type": "string"}}, "required": ["answer", "explanation"]}
```



In [ ]:
agent = create_agent(
    model=ChatOllama(model= "granite4:350m", temperature=0),
    system_prompt=system_prompt,
)

# 7. Define the user input (title + abstract)
title = "The Impact of Climate Change on Biodiversity"
abstract = """
Climate change is altering ecosystems worldwide, leading to shifts in species distributions.
This study explores how rising temperatures and changing precipitation patterns affect biodiversity.
"""

# 8. Invoke the agent with the user prompt
response = agent.invoke({
    "input": f"Title: {title}\nAbstract: {abstract}"
})
response

In [ ]:
# 9. Parse the response into a dictionary
# (Assuming the model returns a list of JSON objects, one per question)
response_text = response["messages"]

# Split the response into parts for each question
response_parts = response_text.strip().split("\n\n")

# Parse each part into the QuestionAnswer schema
answers_dict = {}
for q, question in enumerate(questions):
    question_key = question["name"]
    if q < len(response_parts):
        try:
            parsed = parser.parse(response_parts[q])
            answers_dict[question_key] = {
                "answer": parsed.answer,
                "explanation": parsed.explanation,
            }
        except:
            # Fallback if parsing fails
            answers_dict[question_key] = {
                "answer": "Unknown",
                "explanation": "Failed to parse response.",
            }

# 10. Print the results
for question_key, answer_data in answers_dict.items():
    print(f"{question_key}:")
    print(f"  Answer: {answer_data['answer']}")
    print(f"  Explanation: {answer_data['explanation']}\n")

AttributeError: 'list' object has no attribute 'strip'

In [ ]:
from typing import List, Dict
from pydantic import BaseModel
from langchain.agents import create_agent
from langchain_core.output_parsers import PydanticOutputParser
from langchain_ollama import ChatOllama

# 1. Define the output schema for parsing
class QuestionAnswer(BaseModel):
    answer: str  # "Yes" or "No"
    explanation: str  # 1-2 sentences

# 2. Define the output parser
parser = PydanticOutputParser(pydantic_object=QuestionAnswer)

# 3. Define the list of questions as a dictionary
questions = [
    {"name": "Relevance", "question": "Is the abstract relevant to the title?"},
    {"name": "Clarity", "question": "Is the abstract clear and well-written?"},
    {"name": "Completeness", "question": "Does the abstract cover the main points?"},
]

# 4. Join the questions into a string for the system prompt
questions_str = "\n".join(
    [f"{q['name']}: {q['question']}" for q in questions]
)

# 5. Define the system prompt (static, no mustache)
system_prompt = f"""
You are a precise assistant.
For each of the following questions, answer with a JSON object containing:
- 'answer': 'Yes' or 'No'
- 'explanation': a short explanation (1-2 sentences)
"""

# 6. Create the agent with the static system prompt


{'messages': [AIMessage(content='{\n  "answer": "No",\n  "explanation": "The abstract is not relevant to the title, as it does not address the main content or purpose of the paper."\n}', additional_kwargs={}, response_metadata={'model': 'granite4:350m', 'created_at': '2026-06-24T15:26:30.019086298Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1012606349, 'load_duration': 62326719, 'prompt_eval_count': 255, 'prompt_eval_duration': 20384228, 'eval_count': 39, 'eval_duration': 894444923, 'logprobs': None, 'model_name': 'granite4:350m', 'model_provider': 'ollama'}, id='lc_run--019efa3d-688b-7552-b021-6383bf0b4aca-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 255, 'output_tokens': 39, 'total_tokens': 294})]}

In [ ]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama
from langchain.agents.structured_output import ToolStrategy
from typing import List, Dict, Any
import json

# Define the structured output schema
output_schema = {
    "type": "object",
    "properties": {
        "answer": {
            "type": "string",
            "enum": ["True", "False"],
            "description": "Binary answer: 'True' or 'False'"
        },
        "explanation": {
            "type": "string",
            "description": "A short explanation (1-2 sentences)"
        }
    },
    "required": ["answer", "explanation"]
}

# Initialize ChatOllama
model = ChatOllama(model="granite4:350m", temperature=0)

# Create the agent with structured output
agent = create_agent(
    model=model,
    response_format=ToolStrategy(output_schema),
    system_prompt="You are a precise assistant that analyse the tite and abstract of a paper to answer binary question. Answer with a JSON object containing 'answer' (True/False) and 'explanation' (1-2 sentences)."
)

# Function to process a list of questions
def assess_questions(questions: List[str]) -> List[Dict[str, Any]]:
    results = []
    for question in questions:
        response = agent.invoke({
            "messages": [{"role": "user", "content": question}]
        })
        # Extract the structured response
        structured_response = response.get("structured_response", {})
        results.append(structured_response)
    return results

# Example usage
questions = [
    "Title: The Impact of Climate Change on Biodiversity\nAbstract: Climate change is altering ecosystems worldwide, leading to shifts in species distributions. This study explores how rising temperatures and changing precipitation patterns affect biodiversity.",
    "Title: Quantum Computing in Finance\nAbstract: Quantum computing promises to revolutionize financial modeling by solving complex optimization problems faster than classical computers."
]

# Get results
results = assess_questions(questions)
for result in results:
    print(json.dumps(result, indent=2))

{
  "title": "The Impact of Climate Change on Biodiversity",
  "question": "Does this study focus on the impact of climate change on biodiversity?",
  "answer": "True",
  "explanation": "The study examines how climate change is altering ecosystems worldwide, leading to shifts in species distributions and thus affecting biodiversity."
}
{
  "title": "Quantum Computing in Finance",
  "question": "Is this study about the impact of climate change?",
  "answer": "True",
  "explanation": "The study focuses on quantum computing's potential to revolutionize financial modeling by solving complex optimization problems faster than classical computers, which could have significant implications for the field of finance."
}


In [21]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama
from langchain.agents.structured_output import ToolStrategy
from typing import Annotated, List, Dict, Any, TypedDict
import json
class AnswerWithJustification(TypedDict):
    answer: str
    justification: Annotated[str | None, None, "A justification for the answer."]
# Define the structured output schema
output_schema = {
    "type": "object",
    "properties": {
        "answer": {
            "type": "string",
            "enum": ["True", "False"],
            "description": "Binary answer: 'True' or 'False'"
        },
        "explanation": {
            "type": "string",
            "description": "A short explanation (1-2 sentences)"
        }
    },
    "required": ["answer", "explanation"]
}

# Initialize ChatOllama
model = ChatOllama(model="granite4.1:3b", temperature=0).with_structured_output(AnswerWithJustification)

# Create the agent with structured output
agent = create_agent(
    model=model,
    system_prompt=""""""
).with_str
agent

AttributeError: 'CompiledStateGraph' object has no attribute 'with_str'

In [17]:


# Function to process a list of questions for multiple Title+Abstract pairs
def assess_questions(questions: List[Dict[str, str]], papers: List[Dict[str,str]]) -> List[Dict[str, Any]]:
    results = []
    for p in papers:
        title = p.get("title", "")
        print(title)
        abstract = p.get("abstract", "")
        for q in questions:
            print("  -  ", q)
            question = q.get("question", "")
            # Format the input for the agent
            input_text = f"You are a precise assistant that analyse the tite and abstract of a paper to answer binary question. Answer with a JSON object containing:\n- 'answer': 'True' or 'False'\n- 'justification': a short explanation (1-2 sentences)\nTitle: {title}\nAbstract: {abstract}\nQuestion: {question}"

            response = model.invoke(input_text
                #{"messages": [{"role": "user", "content": input_text}]}
            )
            print(response)
            # Extract the structured response
            structured_response = response.get("structured_response", {})
            results.append({
                "title": title,
                "question": question,
                "question_name" : q.get("name"),
                **structured_response
            })
    return results

questions = [
    {"name": "Relevance", "question": "Is the abstract relevant to the title?"},
    {"name": "Clarity", "question": "Is the abstract clear and well-written?"},
    {"name": "Completeness", "question": "Does the abstract cover the main points?"},
    {"name": "impact_climate_biodiversity", "question": "Does the paper talk about the impact of climate change on biodiversity?"},
    {"name": "impact_climate_human_scoiety", "question": "Does the paper talk about the impact of climate change on human society?"}
]

# Example usage
papers = [
    {
        "title": "Quantum Computing in Finance",
        "abstract": "Quantum computing promises to revolutionize financial modeling by solving complex optimization problems faster than classical computers.",
    },
    {
        "title": "The Impact of Climate Change on Biodiversity",
        "abstract": "Climate change is altering ecosystems worldwide, leading to shifts in species distributions. This study explores how rising temperatures and changing precipitation patterns affect biodiversity.",
    },

]

# Get results
results = assess_questions(questions,papers)
for result in results:
    print(json.dumps(result, indent=2))

Quantum Computing in Finance
  -   {'name': 'Relevance', 'question': 'Is the abstract relevant to the title?'}
{'answer': 'True', 'justification': "The abstract discusses how quantum computing can impact finance through improved financial modeling and problem-solving, which aligns with the title's focus on quantum computing in finance."}
  -   {'name': 'Clarity', 'question': 'Is the abstract clear and well-written?'}
{'answer': 'True', 'justification': 'The abstract concisely states the main benefit of quantum computing in finance without unnecessary detail.'}
  -   {'name': 'Completeness', 'question': 'Does the abstract cover the main points?'}
{'answer': 'True', 'justification': 'The abstract succinctly mentions that quantum computing can revolutionize finance by handling complex optimization problems more efficiently, which directly addresses the main point of its potential impact on financial modeling.'}
  -   {'name': 'impact_climate_biodiversity', 'question': 'Does the paper talk

In [7]:
from slrrag.ollama_analysis import analyze_papers_with_list_agents
from slrrag.settings import PATH_CSV_04_MANUAL_FILTER,PATH_CSV_03_AGENTS_ANWSERS,PATH_CSV_02_FILTERED_TOPIC
from slrrag.input import FIRST_ROUND_QUESTIONS
import pandas as pd
%load_ext autoreload
%autoreload 2
LIST_AGENTS = ["ministral-3:3b","granite4.1:3b"]
df_papers = pd.read_csv(PATH_CSV_02_FILTERED_TOPIC)
df_papers_filtered = df_papers[df_papers.index.isin(df_papers[df_papers["is_reference"]].head(20).index.to_list()) | df_papers.index.isin(df_papers.head(20).index.to_list())]
df_papers_filtered

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


,title,doi,type,year,date,abstract,id_scopus,keywords_scopus,relevance_order_scopus,id_openalex,...,keywords_lens,id_dimensions,topic_dimensions,keywords_dimensions,in_scopus,in_openalex,in_lens,in_dimensions,is_found,is_reference
0,Balancing risks of injury and disturbance to m...,10.1002/2688-8319.12034,article,2020.0,2020-12-01,1. Offshore windfarms require construction pro...,2-s2.0-85110409794,acoustic disturbance | behavioural response | ...,1002.0,https://openalex.org/W3096279247,...,NaN,https://app.dimensions.ai/details/publication/...,B12 Engineering,NaN,True,True,False,True,True,False
1,Co‐designed ecological research for more effec...,10.1002/2688-8319.12130,article,2022.0,2022-01-01,Translational Ecology presents a call for ecol...,NaN,NaN,NaN,https://openalex.org/W4213332547,...,NaN,NaN,NaN,NaN,False,True,True,False,True,False
2,Prevascularized Hydrogel Enhancing Innervation...,10.1002/adhm.202402433,article,2025.0,2025-04-25,Current materials for repairing abdominal peri...,2-s2.0-105000165497,abdominal wall defect repair | biodegradation ...,4456.0,NaN,...,NaN,NaN,NaN,NaN,True,False,False,False,True,False
3,<scp>DARPA</scp>'s explainable<scp>AI</scp>(<s...,10.1002/ail2.61,article,2021.0,2021-12-01,Summary of Defense Advanced Research Projects ...,NaN,NaN,NaN,https://openalex.org/W4200580215,...,NaN,NaN,NaN,NaN,False,True,True,False,True,False
4,Bio‐hybrids: When Robots Come Alive,10.1002/aisy.202500822,article,2025.0,2025-10-01,Bio-hybrid robots are engineered systems that ...,NaN,NaN,NaN,https://openalex.org/W4415468704,...,NaN,NaN,NaN,NaN,False,True,True,False,True,False
5,Floating Offshore Wind Farms Attract Japanese ...,10.1002/aqc.70189,article,2025.0,2025-07-01,Floating offshore wind farms (F-OWFs) are beco...,2-s2.0-105011099858,East China Sea | ecological risk | environment...,647.0,NaN,...,NaN,NaN,NaN,NaN,True,False,False,False,True,False
6,<i><scp>AWWA</scp> Water Science</i> Author Sp...,10.1002/awwa.2051,article,2023.0,2023-03-01,Having recently published an article in AWWA W...,NaN,NaN,NaN,https://openalex.org/W4323318751,...,NaN,NaN,NaN,NaN,False,True,True,False,True,False
7,Evaluation of existing bridges concerning ship...,10.1002/bate.201300026,article,2013.0,2013-05-01,Existing bridges have sometimes to be evaluate...,2-s2.0-84877296944,accidental actions | bridge | reliability | re...,385.0,https://openalex.org/W1981930783,...,NaN,https://app.dimensions.ai/details/publication/...,NaN,NaN,True,True,True,True,True,False
8,Reinforcement of historic bridges crossing the...,10.1002/bate.201500049,article,2015.0,2015-07-01,A change in the ship traffic volume is expecte...,2-s2.0-84934767270,bridges | masonry arches | numerical calculati...,1947.0,https://openalex.org/W2101150443,...,NaN,https://app.dimensions.ai/details/publication/...,NaN,NaN,True,True,True,True,True,False
9,"History of Ecological Sciences, Part 58A: Mari...",10.1002/bes2.1276,article,2016.0,2016-10-01,Part 51 discussed the origins of marine ecolog...,NaN,NaN,NaN,https://openalex.org/W2530684405,...,NaN,NaN,NaN,NaN,False,True,False,False,True,False


In [2]:
FIRST_ROUND_QUESTIONS

[{'name': 'main_question',
  'question': '\nWe consider the following subjects part of risks assessments :\n- Collisions/allisions probabilities\n- Alert system when detecting risky situation\n- Risk or safety index/score calculation\n- Frequency of collisions/allisions\n- Safety distance estimations\n- Explanation and or analysis of collisions using models.\n- Traffic analysis for safety purposes.\n\nBut we are not interested in:\n- Ecological impact (collision between birds and OWF)\n- impact on structural bridges/platforms when collisions. (mainly damage analysis).\n- Structural or design analysis of bridges of offshore infrastructures.\n- Cybersecurity\n- Risk assessment during maintnance operation.\n- Predicting inboard problems.\n- Diverse other risk of accidents related to offshore platforms or bridges (e.g : oil spill, crash during operations, ice)\n- Risk assessment when installing/decommisionning offshore infrastructures. (Impact of weather, ground...)\n- Collision/Risk linke

In [8]:
df_test = analyze_papers_with_list_agents(LIST_AGENTS,df_papers_filtered,questions=FIRST_ROUND_QUESTIONS)
df_test[["title", "is_reference", "main_question_answer_granite4.1:3b", "main_question_justification_granite4.1:3b"]]

---ministral-3:3b----
Balancing risks of injury and disturbance to marine mammals when pile driving at offshore windfarms
---{'name': 'main_question', 'question': '\nWe consider the following subjects part of risks assessments :\n- Collisions/allisions probabilities\n- Alert system when detecting risky situation\n- Risk or safety index/score calculation\n- Frequency of collisions/allisions\n- Safety distance estimations\n- Explanation and or analysis of collisions using models.\n- Traffic analysis for safety purposes.\n\nBut we are not interested in:\n- Ecological impact (collision between birds and OWF)\n- impact on structural bridges/platforms when collisions. (mainly damage analysis).\n- Structural or design analysis of bridges of offshore infrastructures.\n- Cybersecurity\n- Risk assessment during maintnance operation.\n- Predicting inboard problems.\n- Diverse other risk of accidents related to offshore platforms or bridges (e.g : oil spill, crash during operations, ice)\n- Risk a

,title,is_reference,main_question_answer_granite4.1:3b,main_question_justification_granite4.1:3b
0,Balancing risks of injury and disturbance to m...,False,False,The paper focuses on the impacts of pile drivi...
1,Co‐designed ecological research for more effec...,False,False,The provided abstract discusses translational ...
2,Prevascularized Hydrogel Enhancing Innervation...,False,False,The paper focuses on regenerative medicine tec...
3,<scp>DARPA</scp>'s explainable<scp>AI</scp>(<s...,False,False,The abstract discusses the DARPA XAI program f...
4,Bio‐hybrids: When Robots Come Alive,False,False,The provided abstract discusses bio-hybrid rob...
5,Floating Offshore Wind Farms Attract Japanese ...,False,False,The paper focuses on the ecological impact of ...
6,<i><scp>AWWA</scp> Water Science</i> Author Sp...,False,False,The provided abstract discusses risk assessmen...
7,Evaluation of existing bridges concerning ship...,False,False,The paper focuses on evaluating existing bridg...
8,Reinforcement of historic bridges crossing the...,False,False,The paper focuses on the reinforcement of hist...
9,"History of Ecological Sciences, Part 58A: Mari...",False,False,The provided abstract discusses the history of...


In [6]:
df_test.loc[737].abstract

'The developments of offshore wind farms can place increased pressures on conflicting marine users, particularly in already crowded waterways. Risk analysis of potential hazard scenarios are conducted by developers and regulators in the form of Navigation Risk Assessments which seek to identify, measure and mitigate impacts through data collection, consultation, modelling and risk assessment. These activities have inherent uncertainties and limitations which are rarely discussed and have the potential to undermine the value and credibility of the risk assessment. To evaluate the accuracy of Navigation Risk Assessments, their predictions are compared with the historical incident record of accidents involving wind farms. This review identifies significant methodological limitations and sources of uncertainty endemic in the Navigation Risk Assessment process which results in an over-estimation of risk. These include a lack of inclusion of historical evidence, issues during elicitation of 

In [29]:
df_test.dtypes

title                                            str
doi                                              str
type                                             str
year                                         float64
date                                             str
abstract                                         str
id_scopus                                        str
keywords_scopus                                  str
relevance_order_scopus                       float64
id_openalex                                      str
relevance_openalex                           float64
pdf                                              str
topic_openalex                                   str
fields_openalex                                  str
subfields_openalex                               str
domain_openalex                                  str
keywords_openalex                                str
id_lens                                          str
topic_lens                                    